# CNN genre classification

Run this notebook from the updated project. On Colab, upload and extract the project ZIP, then select a GPU runtime if available. You can download GTZAN directly in section 1. Locally, install `requirements-cpu.txt` into your notebook environment first.

This notebook uses the same preprocessing, model and inference code as the web app. Results are computed from your run; recordings, not individual excerpts, define the split.


In [ ]:
from pathlib import Path
import os, sys, subprocess
# Set this to the extracted project folder when using Colab.
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
# Colab example: PROJECT = Path("/content/audio-genre-classifier")
assert (PROJECT / "genre_cnn").is_dir(), "Set PROJECT to the folder containing genre_cnn."
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
import torch
print("Device:", "CUDA GPU" if torch.cuda.is_available() else "CPU")


## 1. Locate GTZAN audio

Set DATA to your existing audio folder, or uncomment the downloader in the next cell to fetch the verified public archive (about 1.2 GB; allow about 3 GB disk space). The folder must contain blues, classical, ... rock.


In [ ]:
# If you do not already have the dataset, uncomment these two lines:
# from genre_cnn.download_data import download
# download("data")
DATA = Path("data/genres")  # Change to your genres_original folder if using Kaggle.
CACHE = Path("data/cnn_cache")
OUTPUT = Path("runs/cnn-reproduction")  # Preserve the bundled trained model.
from genre_cnn.audio import GENRES
assert all((DATA / genre).is_dir() for genre in GENRES), "Check DATA: the ten genre folders must be inside it."


## 2. Prepare and preserve the recording split

Approximately 64/16/20 percent train/validation/test, stratified by genre, seed 42. A saved manifest preserves membership. Corrupt clips and exact decoded-audio duplicates are logged. This is not an artist-disjoint evaluation; near-duplicates and GTZAN annotation issues remain limitations.


In [ ]:
from genre_cnn.prepare import prepare
import json
if (CACHE / "manifest.json").exists():
    manifest = json.loads((CACHE / "manifest.json").read_text())
    print("Using existing split manifest.")
else:
    manifest = prepare(DATA, CACHE)
print({s: sum(r["split"] == s for r in manifest["tracks"]) for s in ["train", "validation", "test"]})
print("Skipped:", len(manifest["skipped"]))


## 3. Train and evaluate

Training-only augmentation, early stopping on validation macro-F1, and a single final test evaluation. Do not keep tuning against the test result. Existing checkpoints are protected; choose a new OUTPUT for a deliberate new experiment. CPU training takes longer; reduce batch size if memory is limited.


In [ ]:
from genre_cnn.train import train
metrics = train(CACHE, OUTPUT, epochs=35, batch_size=32, patience=7, device="auto", threads=2)
print("Held-out recording accuracy:", metrics["test_accuracy"])
print("Held-out recording macro-F1:", metrics["test_macro_f1"])


In [ ]:
from IPython.display import Image, display
import pandas as pd
history = pd.read_json(OUTPUT / "history.json")
display(history.tail())
display(Image(filename=str(OUTPUT / "confusion_matrix.png")))


## 4. Try a new clip

Use a music file that is not part of the training dataset when demonstrating the application. Genre scores are uncalibrated; this model only recognizes the ten GTZAN labels.


In [ ]:
from genre_cnn.predict import Predictor
AUDIO = Path("path/to/your/music.wav")  # Change this.
if AUDIO.is_file():
    print(Predictor(OUTPUT / "cnn.pt").predict(AUDIO))
else:
    print("Set AUDIO to a real WAV, FLAC, OGG, or MP3 clip (at least 3 seconds).")


## 5. Export for deployment

Download the artifact archive before your Colab runtime ends. These are the results of your reproduction run. To deploy them in place of the bundled checkpoint, back up the existing models folder and put the new artifact files there. Keep model, metrics and manifest from the same run together. Follow DEPLOY.md after inspecting your measured results.


In [ ]:
import shutil
archive = shutil.make_archive("cnn-trained-artifacts", "zip", OUTPUT)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Your archive is saved in the project folder.")
